In [1]:
# Libraries
import os
os.environ.pop("POLARS_FEATURE_FLAGS", None)
os.environ["POLARS_SKIP_CPU_CHECK"] = "1"
import svy
import polars as pl
import pyreadstat
import datetime

In [2]:
# Load the survey dataset (Stata .dta) and convert to a Polars DataFrame for svy
df, _meta = pyreadstat.read_dta("../data/survey_data.dta")
smp_data = pl.from_pandas(df)
smp_data = smp_data.with_columns(pl.col("psu").cast(pl.String).cast(pl.Categorical))
print("Rows:", smp_data.height, "Cols:", smp_data.width)
print(smp_data.schema)

Rows: 200 Cols: 8
Schema({'id': Float64, 'psu': Categorical, 'strata': Float64, 'wt_int': Float64, 'sex': Float64, 'age': Float64, 'income': Float64, 'wealth': Float64})


In [3]:
# Set negative values on the outcome (wealth) to missing
print("Negative wealth values:", smp_data.filter(pl.col("wealth") < 0).height)
smp_data = smp_data.with_columns(
    pl.when(pl.col("wealth") < 0).then(None).otherwise(pl.col("wealth")).alias("wealth")
)
print("Missing wealth values:", smp_data["wealth"].null_count())

Negative wealth values: 2
Missing wealth values: 2


In [4]:
# Define the survey design (equivalent to Stata: svyset psu [pw=wt_int], strata(strata))
smp_design = svy.Design(stratum="strata", psu="psu", wgt="wt_int")

# Create the sample object
svy_data = svy.Sample(data=smp_data, design=smp_design)

In [5]:
# Remove records with missing values on the outcome
svy_data_filtered = svy_data.wrangling.filter_records([svy.col("wealth").is_not_null()])
print("Records used:", svy_data_filtered.data.height)

Records used: 198


In [6]:
# Estimate population mean of wealth with design-based standard error
mean_wealth = svy_data_filtered.estimation.mean(y="wealth")
print(mean_wealth)
mean_wealth.to_polars()

╭───────────────── Estimate: MEAN (TAYLOR) ──────────────────╮
│                                                            │
│         est         se          lci          uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  3,792.4663   249.0910   3,261.5415   4,323.3912     6.57  │
│                                                            │
╰────────────────────────────────────────────────────────────╯

est,se,lci,uci,cv
f64,f64,f64,f64,f64
3792.466308,249.090978,3261.541457,4323.391159,0.06568


In [7]:
# Timestamp
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Program completed on: 2026-10-01 20:16:58
